# 03 Feature Engineering

**Input:** `data/processed/02_user_stories_clean.csv` from notebook 02.

**Goal:** turn clean text and clean numbers into features that a scoring rule or a model can read. Each feature answers a small, specific question about a story. For example "does this story start with as a", "does it contain vague words", "is the Story Point on the Fibonacci scale".

**References used in this notebook:**
- Cohn (Mountain Goat Software). The classic template As a / I want / so that and the INVEST checklist.
- Lucassen et al. (2016), QUS framework. Most features map directly to one of the 13 QUS criteria.
- Yamani et al. (2025). LLM-generated user stories evaluated with QUS. Patterns of failure modes I should anticipate (vague words, non-atomic stories, missing reasons).
- Zul et al. (2025). Systematic review listing 8 shared quality criteria across frameworks (Independent, Unambiguous, Complete, Estimable, Testable, Conflict free, Atomic, Negotiable).

**Output:** `data/processed/03_user_stories_features.csv`. Same rows, more columns.

## Plan

1. Load the cleaned CSV.
2. Re run the role, means, and reason detection on the cleaned text.
3. Detect vague words that hurt clarity.
4. Detect implementation hints that violate Problem oriented.
5. Count conjunctions like and, or, & to detect non Atomic stories.
6. Build a few combined features that several QUS criteria depend on.
7. Save the feature enriched CSV.

## Mapping of features to QUS criteria and Cohn's INVEST

| Feature | Maps to | Source |
|---|---|---|
| has_as_a, has_means, has_so_that | QUS Well formed | Cohn template, Lucassen 2016 |
| is_well_formed | QUS Well formed | Lucassen 2016, Zul 2025 |
| is_cohn_full_template | Cohn classic template | Cohn |
| has_vague_words | QUS Unambiguous | Lucassen 2016, Yamani 2025 |
| has_implementation_hint | QUS Problem oriented | Lucassen 2016 |
| has_multi_feature_signal | QUS Atomic, INVEST Independent | Lucassen 2016, Yamani 2025, Cohn INVEST |
| is_fibonacci_sp | QUS Estimatable, INVEST Estimatable | Cohn INVEST |
| description_word_count, title_word_count | QUS Minimal, Full sentence | Lucassen 2016 |

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)
IN_PATH  = Path('../data/processed/02_user_stories_clean.csv')
OUT_PATH = Path('../data/processed/03_user_stories_features.csv')
df = pd.read_csv(IN_PATH, low_memory=False)

for col in ['Creation_Date', 'Resolution_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"Loaded: {IN_PATH}")
print(f"Shape:  {df.shape}")
flag_cols = [c for c in df.columns if c.startswith('flag_')]
print(f"Existing flag columns from notebook 02: {len(flag_cols)}")

for c in flag_cols:
    print(f"  - {c}")

Loaded: ..\data\processed\02_user_stories_clean.csv
Shape:  (31394, 31)
Existing flag columns from notebook 02: 10
  - flag_title_too_short
  - flag_title_too_long
  - flag_description_missing
  - flag_description_too_short
  - flag_description_markup_only
  - flag_sp_non_fibonacci
  - flag_sp_high_scope_risk
  - flag_sp_extreme_scope_risk
  - flag_sp_missing
  - flag_duplicate_in_project


In [2]:
# Group 1: Format markers and well-formed check
# Grounded in:
#   - Cohn (Mountain Goat Software). The classic template:
#       "As a <user>, I want <goal> so that <reason>"
#   - Lucassen et al. (2016), QUS criterion "Well formed":
#       a user story must contain at least a role and a means.
#   - Zul et al. (2025) systematic review confirms that role detection
#     and means detection are the two minimum gates shared by INVEST,
#     QUS, IEEE 830, and USQM frameworks.
#
# Why we recompute these on the cleaned text:
#Notebook 01 ran the same regexes on the RAW text from MySQL.
#Cleaning may have stripped quotes, HTML, and Jira markup, which can change the match boundaries. We want the features to reflect the
#actual text a reader would see, so we run on Title_Clean and Description_Clean.

ROLE_RE   = re.compile(r'\bas an?\b')
MEANS_RE  = re.compile(r'\bi (?:want|need|would like|can|should|am able)\b')
REASON_RE = re.compile(r'\bso that\b')
title_lower = df['Title_Clean'].fillna('').str.lower()
desc_lower  = df['Description_Clean'].fillna('').str.lower()
df['has_as_a']    = title_lower.str.contains(ROLE_RE,   na=False) | desc_lower.str.contains(ROLE_RE,   na=False)
df['has_means']   = title_lower.str.contains(MEANS_RE,  na=False) | desc_lower.str.contains(MEANS_RE,  na=False)
df['has_so_that'] = title_lower.str.contains(REASON_RE, na=False) | desc_lower.str.contains(REASON_RE, na=False)
df['is_well_formed'] = df['has_as_a'] & df['has_means']
df['is_cohn_full_template'] = df['has_as_a'] & df['has_means'] & df['has_so_that']
total = len(df)
print("Group 1 features computed.\n")
print(f"{'Feature':<28} {'True count':>12} {'Share':>10}")
print("-" * 55)
for feat in ['has_as_a', 'has_means', 'has_so_that', 'is_well_formed', 'is_cohn_full_template']:
    n = int(df[feat].sum())
    pct = n / total * 100
    print(f"{feat:<28} {n:>12,} {pct:>9.1f}%")

Group 1 features computed.

Feature                        True count      Share
-------------------------------------------------------
has_as_a                            3,011       9.6%
has_means                           1,923       6.1%
has_so_that                         1,284       4.1%
is_well_formed                      1,427       4.5%
is_cohn_full_template                 321       1.0%


In [3]:
# Group 2: Vague words detection
# Grounded in:
#Lucassen et al. (2016), QUS criterion "Unambiguous":
#a user story should avoid terms that lead to multiple interpretations.
#Yamani et al. (2025) report that vague wording is one of the top
#three failure modes of LLM-generated user stories. About one third of the LLM stories they evaluated were ambiguous.
#Berry and Kamsties (2004), cited inside Lucassen, classify these as "ambiguous adjectives" and "ambiguous adverbs".

# What counts as a vague word:
# Subjective adjectives like "fast", "easy", "good", "robust" that cannot be measured objectively. A QA engineer cannot write a test for "the system should be fast" without further definition.

# vocabulary used in AQUSA (Lucassen et al. 2016).
VAGUE_WORDS = [
    'fast', 'slow', 'easy', 'simple', 'quick',
    'good', 'better', 'best', 'nice', 'great',
    'robust', 'reliable', 'scalable', 'flexible',
    'efficient', 'optimal', 'optimized',
    'user friendly', 'user-friendly', 'intuitive',
    'minimal', 'minimum', 'maximum',
    'appropriate', 'reasonable', 'adequate',
    'modern', 'state of the art', 'state-of-the-art',
    'seamless', 'smooth',
    'high quality', 'high-quality', 'low quality',
    'recent', 'up to date', 'up-to-date',
]

vague_pattern = r'\b(?:' + '|'.join(re.escape(w) for w in VAGUE_WORDS) + r')\b'
VAGUE_RE = re.compile(vague_pattern)
combined_text = (df['Title_Clean'].fillna('') + ' ' + df['Description_Clean'].fillna('')).str.lower()

df['has_vague_words'] = combined_text.str.contains(VAGUE_RE, na=False)
df['vague_word_count'] = combined_text.str.count(VAGUE_RE)
total = len(df)
n_has = int(df['has_vague_words'].sum())
print("Group 2: Vague words")
print("-" * 50)
print(f"Has at least one vague word: {n_has:>6,} ({n_has/total*100:.1f}%)")
print(f"Average vague word count:    {df['vague_word_count'].mean():.2f} per story")
print(f"Max vague word count in one story: {int(df['vague_word_count'].max())}")
print()
from collections import Counter
all_matches = combined_text.str.findall(VAGUE_RE).explode().dropna()
top_vague = Counter(all_matches).most_common(10)
print("Top 10 vague words actually used in this dataset:")
for word, count in top_vague:
    print(f"  {word:<20} {count:>6,}")

Group 2: Vague words
--------------------------------------------------
Has at least one vague word:  4,192 (13.4%)
Average vague word count:    0.18 per story
Max vague word count in one story: 13

Top 10 vague words actually used in this dataset:
  better                  730
  simple                  665
  good                    647
  appropriate             528
  best                    377
  easy                    362
  nice                    307
  recent                  259
  minimum                 210
  quick                   169


In [4]:
# Group 3: Implementation hints (QUS Problem oriented)
# Grounded in:
#   - Lucassen et al. (2016), QUS criterion "Problem oriented":
#     a user story should specify the problem, not the solution.
#   - Zave and Jackson (1997), referenced inside Lucassen, defined the
#     problem-vs-specification distinction that this criterion builds on.
#   - Yamani et al. (2025) found that LLM-generated stories often slip
#     into a solution-oriented voice, especially when the role is
#     "developer" or "researcher" rather than an end user.
# What counts as an implementation hint:
#   Words and phrases that describe HOW the system should be built rather
#   than WHAT the user needs. Examples:
#     - technology mentions: API, REST, SQL, JSON, Python, Lambda
#     - HTTP and protocol terms: POST, GET, HTTP, endpoint
#     - infrastructure: database, server, cluster, queue
#     - UI element prescriptions: button, dropdown, checkbox
#     - code-shape verbs: implement, refactor, configure
#
# Note: This list is intentionally conservative. Some technical words can appear in legitimate user-facing stories (a user *can* care about
#   "downloading a CSV file"). The boolean flag is a signal, not a verdict. The scoring notebook (05) will weight it accordingly.

IMPLEMENTATION_HINT_WORDS = [
    'api', 'rest', 'restful', 'soap', 'graphql',
    'http', 'https', 'tcp', 'udp', 'json', 'xml', 'yaml',
    'post request', 'get request', 'put request', 'delete request',
    'endpoint', 'webhook',
    'sql', 'nosql', 'mongodb', 'mysql', 'postgres', 'postgresql',
    'database', 'schema', 'table', 'column', 'index', 'query',
    'cache', 'redis', 'memcached',
    'server', 'cluster', 'kubernetes', 'docker', 'container',
    'lambda', 'serverless', 'microservice', 'queue', 'kafka', 'rabbitmq',
    'python', 'java', 'javascript', 'typescript', 'golang', 'rust',
    'c++', 'c#',
    'implement', 'refactor', 'configure', 'deploy',
    'click button', 'press button', 'dropdown', 'checkbox', 'radio button',
    'modal', 'popup', 'tooltip',
]

impl_pattern = r'\b(?:' + '|'.join(re.escape(w) for w in IMPLEMENTATION_HINT_WORDS) + r')\b'
IMPL_RE = re.compile(impl_pattern)
df['has_implementation_hint']   = combined_text.str.contains(IMPL_RE, na=False)
df['implementation_hint_count'] = combined_text.str.count(IMPL_RE)
total = len(df)
n_has = int(df['has_implementation_hint'].sum())
print("Group 3: Implementation hints")
print("-" * 50)
print(f"Has at least one implementation hint: {n_has:>6,} ({n_has/total*100:.1f}%)")
print(f"Average implementation hint count:     {df['implementation_hint_count'].mean():.2f} per story")
print(f"Max implementation hints in one story: {int(df['implementation_hint_count'].max())}")
print()

from collections import Counter
impl_matches = combined_text.str.findall(IMPL_RE).explode().dropna()
top_impl = Counter(impl_matches).most_common(10)
print("Top 10 implementation hint words actually used in this dataset:")
for word, count in top_impl:
    print(f"  {word:<20} {count:>6,}")

Group 3: Implementation hints
--------------------------------------------------
Has at least one implementation hint: 13,237 (42.2%)
Average implementation hint count:     1.17 per story
Max implementation hints in one story: 422

Top 10 implementation hint words actually used in this dataset:
  https                 5,489
  java                  3,187
  api                   2,672
  implement             1,907
  table                 1,841
  python                1,800
  http                  1,707
  server                1,408
  query                 1,135
  docker                1,055


In [5]:
# Fix Group 3: strip URLs before counting implementation hints. URLs introduced inflated counts of "http", "https", and bits of
# domain names. URLs themselves are not implementation hints, they are links to context. We remove them, then redo the matching.
URL_RE = re.compile(r'https?://\S+')
combined_text_no_urls = combined_text.str.replace(URL_RE, ' ', regex=True)

IMPLEMENTATION_HINT_WORDS_V2 = [
    'api', 'rest', 'restful', 'soap', 'graphql',
    'tcp', 'udp', 'json', 'xml', 'yaml',
    'post request', 'get request', 'put request', 'delete request',
    'endpoint', 'webhook',
    'sql', 'nosql', 'mongodb', 'mysql', 'postgres', 'postgresql',
    'database', 'schema', 'table', 'column', 'index', 'query',
    'cache', 'redis', 'memcached',
    'server', 'cluster', 'kubernetes', 'docker', 'container',
    'lambda', 'serverless', 'microservice', 'queue', 'kafka', 'rabbitmq',
    'python', 'java', 'javascript', 'typescript', 'golang', 'rust',
    'c++', 'c#',
    'implement', 'refactor', 'configure', 'deploy',
    'click button', 'press button', 'dropdown', 'checkbox', 'radio button',
    'modal', 'popup', 'tooltip',
]

impl_pattern_v2 = r'\b(?:' + '|'.join(re.escape(w) for w in IMPLEMENTATION_HINT_WORDS_V2) + r')\b'
IMPL_RE_V2 = re.compile(impl_pattern_v2)
df['has_implementation_hint']   = combined_text_no_urls.str.contains(IMPL_RE_V2, na=False)
df['implementation_hint_count'] = combined_text_no_urls.str.count(IMPL_RE_V2)
total = len(df)
n_has = int(df['has_implementation_hint'].sum())
print("Group 3 (fixed): Implementation hints, URL-stripped")
print("-" * 50)
print(f"Has at least one implementation hint: {n_has:>6,} ({n_has/total*100:.1f}%)")
print(f"Average implementation hint count:     {df['implementation_hint_count'].mean():.2f} per story")
print(f"Max implementation hints in one story: {int(df['implementation_hint_count'].max())}")
print()

from collections import Counter
impl_matches = combined_text_no_urls.str.findall(IMPL_RE_V2).explode().dropna()
top_impl = Counter(impl_matches).most_common(10)
print("Top 10 implementation hint words (URL-free):")
for word, count in top_impl:
    print(f"  {word:<20} {count:>6,}")

Group 3 (fixed): Implementation hints, URL-stripped
--------------------------------------------------
Has at least one implementation hint: 10,647 (33.9%)
Average implementation hint count:     0.90 per story
Max implementation hints in one story: 395

Top 10 implementation hint words (URL-free):
  java                  3,119
  api                   2,503
  implement             1,907
  table                 1,833
  python                1,592
  server                1,399
  query                 1,122
  docker                  985
  xml                     968
  database                962


In [6]:
# Group 4: Atomicity detection
# Grounded in:
#   - Lucassen et al. (2016), QUS criterion "Atomic":
#     a user story should express exactly one feature. Stories joined
#     by "and" or "or" are usually two stories pretending to be one.
#   - Cohn (Mountain Goat Software), INVEST "Independent":
#     a story should be implementable on its own. Composite stories
#     violate this principle.
#   - Yamani et al. (2025) report that LLM-generated stories with
#     conjunctions are the most common syntactic failure they observed,
#     especially from Gemini (about 25 percent non-atomic).
#   - The AQUSA tool from Lucassen et al. uses the same conjunctions
#     as a primary signal: "and", "or", "&", "+".
#
# What we count:
#   Each conjunction that joins two clauses inside a single story. We count on the cleaned text so HTML and Jira markup are not
#inflating the numbers.


CONJUNCTION_RE = re.compile(r'\band\b|\bor\b|&|\+')
df['conjunction_count'] = combined_text_no_urls.str.count(CONJUNCTION_RE)
df['has_multi_feature_signal'] = df['conjunction_count'] > 3

total = len(df)
n_has = int(df['has_multi_feature_signal'].sum())
print("Group 4: Atomicity")
print("-" * 50)
print(f"Has multi-feature signal (>3 conjunctions): {n_has:>6,} ({n_has/total*100:.1f}%)")
print(f"Mean conjunction count:    {df['conjunction_count'].mean():.2f}")
print(f"Median conjunction count:  {df['conjunction_count'].median():.0f}")
print(f"Max conjunction count:     {int(df['conjunction_count'].max())}")
print()
print("Distribution of conjunction count:")
bins = [0, 1, 2, 4, 6, 10, 20, 9999]
labels = ['0', '1', '2-3', '4-5', '6-9', '10-19', '20+']
df['_conj_bin'] = pd.cut(df['conjunction_count'], bins=bins, labels=labels, right=False, include_lowest=True)
print(df['_conj_bin'].value_counts().sort_index())
df = df.drop(columns=['_conj_bin'])

Group 4: Atomicity
--------------------------------------------------
Has multi-feature signal (>3 conjunctions):  2,973 (9.5%)
Mean conjunction count:    1.34
Median conjunction count:  1
Max conjunction count:     68

Distribution of conjunction count:
_conj_bin
0        13507
1         7988
2-3       6926
4-5       1854
6-9        846
10-19      240
20+         33
Name: count, dtype: int64


In [7]:
# Group 5: Story Point as positive features
# Grounded in:
#   - Cohn (Mountain Goat Software), INVEST criterion "Estimatable":
#     a story must be estimable. Missing or extreme estimates fail this.
#   - Cohn, Planning Poker. Teams that follow the practice estimate on
#     the modified Fibonacci scale (1, 2, 3, 5, 8, 13, 21, 34, 55, 89).
#   - Lucassen et al. (2016), QUS criterion "Estimatable".
#
# Notebook 02 already added negative flags (flag_sp_missing,
# flag_sp_non_fibonacci, flag_sp_high_scope_risk, flag_sp_extreme_scope_risk).
# Here I added positive mirrors that are easier to read in scoring rules.

df['is_fibonacci_sp'] = ~df['flag_sp_non_fibonacci'] & ~df['flag_sp_missing']
df['has_sp'] = ~df['flag_sp_missing']
df['sp_within_sprint_range'] = df['has_sp'] & ~df['flag_sp_high_scope_risk']

print("Group 5: Story Point positive features")
print("-" * 50)
total = len(df)
for feat in ['has_sp', 'is_fibonacci_sp', 'sp_within_sprint_range']:
    n = int(df[feat].sum())
    print(f"{feat:<28} {n:>6,} ({n/total*100:5.1f}%)")

Group 5: Story Point positive features
--------------------------------------------------
has_sp                       21,733 ( 69.2%)
is_fibonacci_sp              14,233 ( 45.3%)
sp_within_sprint_range       20,639 ( 65.7%)


In [8]:
# Group 6: Acceptance criteria signal
# Grounded in:
#   - Cohn (Mountain Goat Software), "Conditions of Satisfaction":
#     a short list of checks that must be true for the story to be
#     accepted. Also called acceptance criteria.
#   - Ron Jeffries (2001), the "Confirmation" pillar of the three Cs.
# What we look for: Literal phrases like "acceptance criteria", "criteria:", "AC:"GIVEN-WHEN-THEN structure (BDD style) Checklist markers like "- [ ]" or "[x]"

AC_PHRASE_RE = re.compile(
    r'\bacceptance criteria\b'
    r'|\bac:\b'
    r'|\bcriteria:\b'
    r'|\bgiven\b.{0,80}\bwhen\b.{0,80}\bthen\b',
    re.DOTALL
)
AC_CHECKLIST_RE = re.compile(r'\[\s*[x ]\s*\]')

df['has_acceptance_criteria'] = (
    combined_text_no_urls.str.contains(AC_PHRASE_RE, na=False)
    | combined_text_no_urls.str.contains(AC_CHECKLIST_RE, na=False)
)

total = len(df)
n = int(df['has_acceptance_criteria'].sum())
print("Group 6: Acceptance criteria signal")
print("-" * 50)
print(f"has_acceptance_criteria  {n:>6,} ({n/total*100:.1f}%)")

Group 6: Acceptance criteria signal
--------------------------------------------------
has_acceptance_criteria     228 (0.7%)


In [9]:
# Summary of all features added in notebook 03

feature_groups = {
    'Group 1 - Format markers (Cohn / QUS Well-formed)': [
        'has_as_a', 'has_means', 'has_so_that',
        'is_well_formed', 'is_cohn_full_template',
    ],
    'Group 2 - Vague words (QUS Unambiguous)': [
        'has_vague_words',
    ],
    'Group 3 - Implementation hints (QUS Problem-oriented)': [
        'has_implementation_hint',
    ],
    'Group 4 - Atomicity (QUS Atomic / INVEST Independent)': [
        'has_multi_feature_signal',
    ],
    'Group 5 - Story Point (INVEST Estimatable)': [
        'has_sp', 'is_fibonacci_sp', 'sp_within_sprint_range',
    ],
    'Group 6 - Acceptance criteria (Cohn Conditions of Satisfaction)': [
        'has_acceptance_criteria',
    ],
}

total = len(df)
print(f"Feature summary across {total:,} stories\n")

for group, feats in feature_groups.items():
    print(f"\n{group}")
    print("-" * len(group))
    for feat in feats:
        if feat in df.columns:
            n = int(df[feat].sum())
            print(f"  {feat:<32} {n:>6,} ({n/total*100:5.1f}%)")
        else:
            print(f"  {feat:<32} MISSING")

print("\nNumeric companion features")
print("-" * 26)
for feat in ['vague_word_count', 'implementation_hint_count', 'conjunction_count']:
    if feat in df.columns:
        print(f"  {feat:<32} mean={df[feat].mean():.2f}  median={df[feat].median():.0f}  max={int(df[feat].max())}")

Feature summary across 31,394 stories


Group 1 - Format markers (Cohn / QUS Well-formed)
-------------------------------------------------
  has_as_a                          3,011 (  9.6%)
  has_means                         1,923 (  6.1%)
  has_so_that                       1,284 (  4.1%)
  is_well_formed                    1,427 (  4.5%)
  is_cohn_full_template               321 (  1.0%)

Group 2 - Vague words (QUS Unambiguous)
---------------------------------------
  has_vague_words                   4,192 ( 13.4%)

Group 3 - Implementation hints (QUS Problem-oriented)
-----------------------------------------------------
  has_implementation_hint          10,647 ( 33.9%)

Group 4 - Atomicity (QUS Atomic / INVEST Independent)
-----------------------------------------------------
  has_multi_feature_signal          2,973 (  9.5%)

Group 5 - Story Point (INVEST Estimatable)
------------------------------------------
  has_sp                           21,733 ( 69.2%)
  is_fibonacci_

In [10]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False, encoding='utf-8')
import os
size_mb = os.path.getsize(OUT_PATH) / 1024**2

print(f"Saved: {OUT_PATH}")
print(f"Rows:  {len(df):,}")
print(f"Cols:  {len(df.columns)}")
print(f"Size:  {size_mb:.1f} MB on disk")
feature_cols = [c for c in df.columns if c.startswith(('has_', 'is_', 'sp_within_'))]
count_cols   = [c for c in df.columns if c.endswith('_count')]
flag_cols    = [c for c in df.columns if c.startswith('flag_')]
print(f"\nBoolean feature columns:    {len(feature_cols)}")
print(f"Numeric companion columns:  {len(count_cols)}")
print(f"Original flag columns (notebook 02): {len(flag_cols)}")

Saved: ..\data\processed\03_user_stories_features.csv
Rows:  31,394
Cols:  46
Size:  42.5 MB on disk

Boolean feature columns:    13
Numeric companion columns:  7
Original flag columns (notebook 02): 10


## Summary

**What this notebook did:**
- Loaded the cleaned dataset (31,394 stories, 31 columns) from notebook 02.
- Added 13 new boolean features and 3 numeric companion features. Total columns now 46.
- Every feature maps to a specific quality criterion from QUS, INVEST, or Cohn's classic template.
- Saved the result to `data/processed/03_user_stories_features.csv`.

**Feature groups and their academic grounding:**

| Group | Features | Anchored to |
|---|---|---|
| 1. Format markers | has_as_a, has_means, has_so_that, is_well_formed, is_cohn_full_template | Cohn classic template, Lucassen et al. (2016) QUS "Well formed" |
| 2. Vague words | has_vague_words, vague_word_count | Lucassen et al. (2016) QUS "Unambiguous", Yamani et al. (2025) on LLM failure modes |
| 3. Implementation hints | has_implementation_hint, implementation_hint_count | Lucassen et al. (2016) QUS "Problem oriented", traces back to Zave and Jackson (1997) |
| 4. Atomicity | has_multi_feature_signal, conjunction_count | Lucassen et al. (2016) QUS "Atomic", Cohn INVEST "Independent" |
| 5. Story Point | has_sp, is_fibonacci_sp, sp_within_sprint_range | Cohn (Planning Poker, INVEST "Estimatable"), Lucassen et al. (2016) |
| 6. Acceptance criteria | has_acceptance_criteria | Cohn "Conditions of Satisfaction", Jeffries (2001) three Cs |

**Highlights for the case study:**

- Only 1.0% of stories follow the full Cohn template after cleaning (confirms Finding 9).
- 4.5% pass the QUS "Well formed" gate (role + means together).
- 13.4% contain at least one vague word. Top offenders: "better", "simple", "good", "appropriate", "best".
- 33.9% contain an implementation hint. This is high but expected for an open source software dataset where teams are deeply technical. Notebook 05 will weight this signal cautiously.
- Only 9.5% have a strong multi-feature signal (more than 3 conjunctions). Most stories are atomic in shape, though not necessarily in meaning.
- About one in three stories with an estimate uses a non Fibonacci value, confirming the planning poker discipline gap from Finding 17.
- **Only 0.7% mention acceptance criteria explicitly** (Finding 18). This is the rarest quality signal in the dataset and will be one of the strongest weights in the scoring model.

**What is next:** notebook 04 turns these raw features into the actual quality framework. It will define dimension scores (clarity, completeness, testability, business value, scope risk) by combining the features above with the flags from notebook 02, all grounded in the 13 QUS criteria.